# Per-cell Q-trajectories — every disagreement, per run

For each run that logged the full Q-grid (`--log-q-grid`), this draws a grid of small charts — one
per *genuine-disagreement* cell — each showing Q(hit/stand/double) over training. Lets you see which
cells **oscillate** (the high-variance double instability), which are **stable-but-wrong**, and which
slowly drift.

The 27 sweep runs do NOT have `q_grid` (only the 5 fixed probe cells). Re-run the run(s) you want
to inspect with `--log-q-grid`, e.g.:

    python -m blackjack_rl.dqn_experiment --episodes 1000000 --epsilon-schedule linear \
        --epsilon-start 0.3 --epsilon-end 0.0 --eval-hands 50000 --seed 42 --encoding onehot --log-q-grid


In [ ]:
import json, glob, os
from pathlib import Path
from math import ceil
import numpy as np
import matplotlib.pyplot as plt

RUNS = Path("..") / "runs"
RECORDS = []
for f in sorted(RUNS.glob("*/record.json")):
    try:
        r = json.load(open(f)); r["_id"] = f.parent.name; RECORDS.append(r)
    except Exception as e:
        print("skip", f, e)
is_dqn = lambda r: r.get("method") == "dqn"
have_grid = [r for r in RECORDS if is_dqn(r) and any("q_grid" in p for p in r.get("learning_curve", []))]
print(f"{len(RECORDS)} records | {len(have_grid)} have a full q_grid")
for r in have_grid:
    c = r["config"]
    print(f"  {r['_id']}  enc={c.get('encoding')} double={c.get('double_dqn')} ES={c.get('exploring_starts')} lr={c.get('lr')} seed={c.get('seed')}")

## The plotting function

In [ ]:
ACT_COLOR = {"hit": "tab:blue", "stand": "tab:orange", "double": "tab:green", "split": "tab:red"}

def cell_label(c):
    return f"{'soft' if c['is_soft'] else 'hard'}{c['player_value']}_v{c['dealer_upcard']}"

def plot_run_disagreements(r, cols=5, max_cells=60):
    grid_pts = [p for p in r.get("learning_curve", []) if "q_grid" in p]
    if not grid_pts:
        print(f"run {r['_id']} has no q_grid — re-run it with --log-q-grid"); return
    eps = [p["episode"] for p in grid_pts]
    dis = [c for c in r["diff"]["cells"] if c["category"] == "genuine_disagreement"][:max_cells]
    if not dis:
        print(f"run {r['_id']} has no genuine disagreements"); return
    n = len(dis); rows = ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 2.4 * rows), sharex=True)
    axes = np.array(axes).reshape(-1)
    actions = list(next(iter(grid_pts[-1]["q_grid"].values())).keys())
    for ax, c in zip(axes, dis):
        lab = cell_label(c)
        for act in actions:
            ys = [p["q_grid"].get(lab, {}).get(act) for p in grid_pts]
            ax.plot(eps, ys, color=ACT_COLOR.get(act, "gray"), lw=1.1, label=act)
        ax.set_title(f"{lab}  net={c['agent_action'][0].upper()} bas={c['basic_action'][0].upper()}", fontsize=8)
        ax.grid(alpha=.3); ax.tick_params(labelsize=6); ax.axhline(0, color="k", lw=.4, alpha=.4)
    for ax in axes[n:]:
        ax.axis("off")
    axes[0].legend(fontsize=7, loc="best")
    cfg = r["config"]
    fig.suptitle(
        f"Q-trajectories — all {n} disagreements | {r['_id']} "
        f"(enc={cfg.get('encoding')} double={cfg.get('double_dqn')} ES={cfg.get('exploring_starts')} "
        f"lr={cfg.get('lr')} seed={cfg.get('seed')})  agree={r['diff']['agreement_unweighted']*100:.1f}%",
        y=1.005, fontsize=11,
    )
    plt.tight_layout(); plt.show()

## One grid per run (all runs that have a q_grid)

In [ ]:
if not have_grid:
    print("No runs with q_grid yet. Re-run with --log-q-grid (see the intro cell), then re-run this notebook.")
for r in have_grid:
    plot_run_disagreements(r)

## Read the grids like this
- **Oscillating `double` (green swinging widely)** while hit/stand are flat -> the high-variance
  terminal-action instability; the final action is just whatever phase the swing was in.
- **Stable-but-wrong** (lines flat, wrong one on top) -> a genuine value error the net settled on.
- **Slow crossing near the end** -> a near-tie boundary cell that hadn't converged.
- Compare a natural-play run to its exploring-starts twin: did forced coverage *calm* the green
  oscillation, or just shift which cells wobble?